# Notebook 2 — VLM & Semantic Layer Testing

Tests the **Macro-Loop** (low-frequency semantic abstraction):

| # | Module | What it tests |
|---|--------|---------------|
| 1 |  | Parse a mock VLM string into typed SPO triples via Ollama |
| 2 |  | Insert event chunks + semantic similarity search in Milvus |
| 3 |  | Full Qwen2.5-VL-3B scene graph on a synthetic image (GPU required) |

> **Cell 3 requires** the Qwen2.5-VL-3B-Instruct model and a GPU. Cells 1-2 run on CPU.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
print("sys.path ready")

## 1. EntityExtractor — Parse VLM text into typed SPO triples

Requires Ollama running with  pulled.

In [ ]:
from src.semantic_abstractor.entity_extractor import EntityExtractor

# Check Ollama is reachable before loading the model
import subprocess, sys
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if "qwen2.5:72b" not in result.stdout:
    print("WARNING: qwen2.5:72b not found in ollama list. Pull it with: ollama pull qwen2.5:72b")
    print("Skipping EntityExtractor test.")
else:
    extractor = EntityExtractor(model_name="qwen2.5:72b")
    mock_vlm = "Vehicle 4 is following Vehicle 9 very closely at high speed. Vehicle 2 stops at a red traffic light. A pedestrian is crossing in front of Vehicle 3.", "Vehicle 2 stops at a red traffic light. A pedestrian is crossing in front of Vehicle 3."

    triples = extractor.extract_triples(mock_vlm, current_time=12.5)

    import json
    print("Extracted triples:")
    print(json.dumps(triples, indent=2))

    assert isinstance(triples, list), "Expected a list of triples"
    assert len(triples) > 0, "Expected at least one triple"

    # Verify Pydantic-enforced types
    valid_types = {"Vehicle", "Pedestrian", "Infrastructure"}
    for t in triples:
        assert t.get("subject_type") in valid_types, f"Bad subject_type: {t}"
        assert t.get("object_type")  in valid_types, f"Bad object_type: {t}"

    print(f"PASS: EntityExtractor returned {len(triples)} typed SPO triples")

## 2. SemanticVectorStore — Insert & Search

Insert 5 synthetic VLM event descriptions, then run 2 semantic queries and verify top-1 relevance.

In [ ]:
import shutil
from src.memory_layer.milvus_client import SemanticVectorStore

TEST_MILVUS = "/tmp/test_milvus_semantic.db"
if os.path.exists(TEST_MILVUS): os.remove(TEST_MILVUS)

# Pre-canned scene descriptions (simulate 5 VLM output chunks)
event_chunks = [
    ("Vehicle 4 tailgating Vehicle 9 at 25 m/s on the highway.",          0.0,  5.0),
    ("Vehicle 2 braking hard at the intersection; near-miss with Vehicle 7.", 5.0, 10.0),
    ("Pedestrian crossing in front of Vehicle 3 on the zebra crossing.",    10.0, 15.0),
    ("Vehicle 6 running a red light at the traffic light.",                 15.0, 20.0),
    ("Vehicle 1 and Vehicle 5 lane-change conflict on the motorway.",       20.0, 25.0),
]

store = SemanticVectorStore(db_path=TEST_MILVUS)
for desc, t_start, t_end in event_chunks:
    store.insert_event_chunk(desc, t_start, t_end)
    print(f"  Inserted [{t_start:.0f}-{t_end:.0f}s]: {desc[:60]}...")

print(f"Inserted {len(event_chunks)} events into Milvus Lite")

In [ ]:
import json

# Query 1: physics / speed
q1 = "Which vehicle was speeding?"
r1 = store.search_semantic_events(q1, top_k=2)
print(f"Query: "{q1}"")
print(json.dumps(r1, indent=2))

# Query 2: pedestrian
q2 = "pedestrian near-miss"
r2 = store.search_semantic_events(q2, top_k=2)
print(f"Query: "{q2}"")
print(json.dumps(r2, indent=2))
assert len(r1) > 0 and len(r2) > 0

# Top result for pedestrian query should mention "Pedestrian" or "crossing"
top_desc = r2[0].get("description", "")
print(f"Top match for pedestrian query: {top_desc}")
assert "Pedestrian" in top_desc or "crossing" in top_desc, "Unexpected top result"
print("PASS: SemanticVectorStore returns relevant results for both queries")

store.close()

## 3. VLM Inference — Qwen2.5-VL scene graph on a synthetic image

> **Requires GPU and Qwen2.5-VL-3B-Instruct model.**  
> Skip this cell if running on CPU only.

In [ ]:
import torch
from PIL import Image
import numpy as np
import cv2
from src.semantic_abstractor.set_of_mark import AdaptiveRenderer, RenderContext
from src.semantic_abstractor.vlm_inference import TrafficSemanticAbstractor

if not torch.cuda.is_available():
    print("No CUDA GPU detected — skipping VLM inference test.")
else:
    # Build a synthetic SoM image with 3 vehicles
    frame = np.full((720, 1280, 3), 50, dtype=np.uint8)
    fake_tracks = np.array([[100,200,250,320,4],[500,300,650,420,9],[900,100,1000,200,3]])
    renderer = AdaptiveRenderer()
    ctx = RenderContext()
    ctx.update(fake_tracks, timestamp=10.0)
    renderer.render(frame, ctx)
    som_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    # Mock physics state vectors to inject into VLM prompt
    state_vectors = {
        4: [10.0, 5.0, 12.5, 0.1, -1.2, 0.0],
        9: [30.0, 8.0,  8.0, 0.2,  0.5, 0.0],
        3: [50.0, 3.0, 15.0, 0.0,  2.0, 0.0],
    }

    vlm = TrafficSemanticAbstractor(model_id="Qwen/Qwen2.5-VL-3B-Instruct")
    triples = vlm.generate_scene_graph_triples(som_pil, timestamp=10.0, state_vectors=state_vectors)

    print("VLM raw output:")
    import json
    print(json.dumps(triples, indent=2))

    assert isinstance(triples, list), "Expected list from VLM"
    print(f"PASS: VLM returned {len(triples)} SPO triples from synthetic image")